# Poisson Regression for the AAV Linear Model

This notebook implements Poisson GLM as an alternative to Ridge regression on log-ratios.
It follows the two-step protocol:
1. **Viability** : $n^{(0)} \to \lambda_1'$ — Poisson GLM with offset $\log n_n^{(0)}$
2. **Selectivity** : $\lambda_1' \to n^{(2)}$ — Poisson GLM with offset $\log \lambda_{1,n}'$

See `Poisson_regression_AAV.tex` for the mathematical justification.

## 1. Imports & shared setup

Reuses `sequences`, `n0`, `lambda1p`, `n2`, `X_bias`, `T_viab`, `T_sel` from `first_ML_model.ipynb`.
Run that notebook first, or regenerate the data below.

In [3]:
import numpy as np
import jax
import jax.numpy as jnp
import statsmodels.api as sm
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sequence_classes import protocol, calculate_scores, T_sel, T_viab
from analysis import pearson, precision_at_k

key = jax.random.key(42)
key_seq, key_viab_w, key_sel_w, key_n0, key_run = jax.random.split(key, 5)

num_amino_acids = 20
num_positions   = 7
num_sequences   = 1_000_000

sequences = jax.random.randint(
    key_seq, shape=(num_sequences, num_positions),
    minval=0, maxval=num_amino_acids
)

n0 = jax.random.randint(
    key_n0, shape=(num_sequences,), minval=10, maxval=1000
).astype(jnp.float32)

viability_weights_GT   = jax.random.normal(key_viab_w, shape=(num_amino_acids, num_positions))
selectivity_weights_GT = jax.random.normal(key_sel_w,  shape=(num_amino_acids, num_positions))

_, capsids, lambda1p, n2 = protocol(
    n0, sequences,
    viability_weights_GT,
    selectivity_weights_GT,
    use_full_ngs=False
)

X_oh   = jax.nn.one_hot(sequences, num_amino_acids)
X_flat = np.array(X_oh.reshape(num_sequences, -1))
X_bias = np.hstack([X_flat, np.ones((num_sequences, 1))])

ImportError: cannot import name 'pearson' from 'analysis' (/Users/merlincc/Library/Mobile Documents/com~apple~CloudDocs/IDV AAV/Directed-Evolution-loop/analysis.py)

## 2. Shared utilities

In [ ]:
def extract_weights(glm_result, T):
    """
    Reshape the GLM coefficient vector back to a (20, 7) weight matrix.
    The last coefficient is the intercept → drop it.
    Multiply by T to recover the scale of the original score function.
    """
    coef = glm_result.params
    w = coef[:-1].reshape(num_positions, num_amino_acids).T
    return jnp.array(w * T)


## 3. Viability — Poisson GLM

### Model

$$
\lambda_{1,n}' \sim \mathrm{Poisson}\!\left(\exp\!\left(\log n_n^{(0)} + \frac{\mathbf{x}_n^\top \mathbf{w}^\mathrm{viab}}{T_\mathrm{viab}} + c\right)\right)
$$

The offset $\log n_n^{(0)}$ is known and fixed. `statsmodels` accepts it via the `offset` argument of `GLM`.

Zero-count sequences ($\lambda_{1,n}' = 0$) are **included** — they contribute $-\mu_n$ to the log-likelihood and push the predicted rate down for low-viability variants.  
No masking or pseudocount is needed.


In [ ]:
# ── Offset ────────────────────────────────────────────────────────────────────
offset_viab = np.log(np.array(n0, dtype=float) + 0.5)   # +0.5 avoids log(0) for n0=0

# ── Fit ───────────────────────────────────────────────────────────────────────
glm_viab = sm.GLM(
    endog  = np.array(lambda1p, dtype=float),
    exog   = X_bias,                            # (N, 141)  one-hot + bias column
    family = sm.families.Poisson(link=sm.families.links.Log()),
    offset = offset_viab,
)
# TODO: add regularisation (alpha parameter) once you validate the unpenalised fit
result_viab = glm_viab.fit(maxiter=50, tol=1e-6)
print(result_viab.summary2().tables[0])         # convergence info


NameError: name 'n0' is not defined

In [ ]:
viability_weights_poisson = extract_weights(result_viab, T_viab)

v_gt = calculate_scores(sequences, viability_weights_GT)
v_hat_poisson = calculate_scores(sequences, viability_weights_poisson)

print(f'Pearson r (scores) : {pearson(v_gt, v_hat_poisson):.4f}')
print(f'Pearson r (weights): {pearson(np.array(viability_weights_poisson).flatten(), np.array(viability_weights_GT).flatten()):.4f}')


## 4. Regularised Poisson Viability (L2 penalty)

`sm.GLM.fit_regularized` adds an $\ell_2$ penalty $\lambda \|\mathbf{w}\|^2$ to the Poisson log-likelihood.  
The IRLS update becomes $(X^\top W X + \lambda I)^{-1} X^\top W z$ — same structure as Ridge.  
Select $\lambda$ by 5-fold cross-validation on the **Poisson deviance** rather than MSE.

$$
D(y, \hat{\mu}) = 2 \sum_n \left[ y_n \log\frac{y_n}{\hat{\mu}_n} - (y_n - \hat{\mu}_n) \right]
$$


In [ ]:
# ── Cross-validation on Poisson deviance ──────────────────────────────────────
lambdas  = np.logspace(-2, 2, 30)
K_FOLDS  = 5
kf       = KFold(n_splits=K_FOLDS, shuffle=True, random_state=0)

# TODO: implement CV loop
# Hint: for each (train, val) split and each lam:
#   fit sm.GLM(...).fit_regularized(alpha=lam, L1_wt=0.0)  ← pure L2
#   compute Poisson deviance on the validation fold
#   average across folds

cv_deviance_viab = np.zeros(len(lambdas))   # fill this in

lam_best_viab = lambdas[np.argmin(cv_deviance_viab)]
print(f'Best λ viabilité : {lam_best_viab:.3f}')

# ── Plot CV curve ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogx(lambdas, cv_deviance_viab, color='steelblue', lw=2)
ax.axvline(lam_best_viab, color='k', linestyle='--', lw=1, label=f'best λ = {lam_best_viab:.3f}')
ax.set_xlabel('λ'); ax.set_ylabel(f'{K_FOLDS}-fold Poisson deviance')
ax.set_title('Viabilité — CV deviance vs λ')
ax.legend(); ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout(); plt.show()


In [ ]:
# ── Refit with best λ ────────────────────────────────────────────────────────
result_viab_reg = sm.GLM(
    endog  = np.array(lambda1p, dtype=float),
    exog   = X_bias,
    family = sm.families.Poisson(link=sm.families.links.Log()),
    offset = offset_viab,
).fit_regularized(alpha=lam_best_viab, L1_wt=0.0, maxiter=100)

viability_weights_poisson = extract_weights(result_viab_reg, T_viab)
v_hat_poisson = calculate_scores(sequences, viability_weights_poisson)
print(f'Pearson r (scores) : {pearson(v_gt, v_hat_poisson):.4f}')


## 5. Selectivity — Poisson GLM

### Standard approach (noisy offset)

Treat the observed $\lambda_{1,n}'$ as a fixed offset:
$$
n_n^{(2)} \sim \mathrm{Poisson}\!\left(\exp\!\left(\log \lambda_{1,n}' + \frac{\mathbf{x}_n^\top \mathbf{w}^\mathrm{sel}}{T_\mathrm{sel}} + c\right)\right)
$$

**Caveat**: sequences with $\lambda_{1,n}' = 0$ cannot be included (offset = $-\infty$). They must be masked — same as Ridge.  
The errors-in-variables bias is largest for sequences with few viability reads.


In [ ]:
# ── Mask: only sequences with at least one viability read ────────────────────
mask_sel   = np.array(lambda1p > 0)
X_sel      = X_bias[mask_sel]
n2_sel     = np.array(n2, dtype=float)[mask_sel]
l1p_sel    = np.array(lambda1p, dtype=float)[mask_sel]

offset_sel = np.log(l1p_sel)   # noisy offset

# ── Fit ───────────────────────────────────────────────────────────────────────
# TODO: cross-validate λ (same structure as Section 4, but on selectivity data)
lam_best_sel = 1.0   # placeholder — replace with CV result

result_sel = sm.GLM(
    endog  = n2_sel,
    exog   = X_sel,
    family = sm.families.Poisson(link=sm.families.links.Log()),
    offset = offset_sel,
).fit_regularized(alpha=lam_best_sel, L1_wt=0.0, maxiter=100)

selectivity_weights_poisson = extract_weights(result_sel, T_sel)


## 6. Selectivity — Smoothed Offset (recommended)

Instead of using the noisy $\log \lambda_{1,n}'$ as offset, use the **predicted** log-rate from the viability fit:
$$
\hat{o}_n = \log n_n^{(0)} + \frac{\hat{f}_n^{\mathrm{viab}}}{T_\mathrm{viab}}
$$
This eliminates the Poisson noise from the offset and allows all $N$ sequences to be included (even those with $\lambda_{1,n}' = 0$).  
It requires one additional Poisson fit but reduces the errors-in-variables bias.


In [ ]:
# ── Smoothed viability offset ─────────────────────────────────────────────────
v_hat_scores  = np.array(calculate_scores(sequences, viability_weights_poisson))  # f_n_viab / T_viab * T_viab
# The linear predictor (log-rate without intercept) is:
offset_smooth = np.log(np.array(n0, dtype=float) + 0.5) + v_hat_scores / T_viab
# Note: result_viab_reg.predict() gives µ_n; take log to get the linear predictor
# offset_smooth = np.log(result_viab_reg.predict() + 1e-9)   # alternatively

# ── Fit selectivity on all sequences ─────────────────────────────────────────
# TODO: same GLM structure as above but with offset_smooth and all N sequences
# result_sel_smooth = sm.GLM(...).fit_regularized(...)
# selectivity_weights_smooth = extract_weights(result_sel_smooth, T_sel)


## 7. Weight recovery & Precision@k

In [ ]:
s_gt = calculate_scores(sequences, selectivity_weights_GT)
s_hat_poisson = calculate_scores(sequences, selectivity_weights_poisson)

combined_gt  = np.array(v_gt)  / T_viab + np.array(s_gt)  / T_sel
combined_hat = np.array(v_hat_poisson) / T_viab + np.array(s_hat_poisson) / T_sel

fracs = [0.01, 0.05, 0.10, 0.20]
print('  Top-%    Precision@k   (random baseline = top-%)')
print('  ──────────────────────────────────────────────────')
for frac in fracs:
    p = precision_at_k(combined_gt, combined_hat, k_frac=frac)
    print(f'  Top {int(frac*100):3d}%      {p:.3f}          (baseline: {frac:.3f})')


In [ ]:
# ── Score scatter ─────────────────────────────────────────────────────────────
k10     = int(num_sequences * 0.10)
top_gt  = set(np.argsort(-combined_gt)[:k10])
top_hat = set(np.argsort(-combined_hat)[:k10])
both    = np.array(sorted(top_gt & top_hat))
only_gt = np.array(sorted(top_gt - top_hat))
only_hat= np.array(sorted(top_hat - top_gt))
rest    = np.array(sorted(set(range(num_sequences)) - top_gt - top_hat))

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(combined_gt[rest],     combined_hat[rest],     s=4,  alpha=0.2, color='lightgray')
ax.scatter(combined_gt[only_hat], combined_hat[only_hat], s=12, alpha=0.6, color='darkorange', label=f'False pos. ({len(only_hat)})')
ax.scatter(combined_gt[only_gt],  combined_hat[only_gt],  s=12, alpha=0.6, color='steelblue',  label=f'Missed ({len(only_gt)})')
ax.scatter(combined_gt[both],     combined_hat[both],     s=12, alpha=0.8, color='forestgreen', label=f'Recovered ({len(both)})')
lo, hi = combined_gt.min(), combined_gt.max()
ax.plot([lo, hi], [lo, hi], 'k--', lw=1)
thr_gt  = np.sort(combined_gt)[-k10]
thr_hat = np.sort(combined_hat)[-k10]
ax.axvline(thr_gt,  color='steelblue',  linestyle=':', lw=1.2)
ax.axhline(thr_hat, color='darkorange', linestyle=':', lw=1.2)
ax.set_xlabel('GT combined score'); ax.set_ylabel('Predicted combined score (Poisson)')
p10 = precision_at_k(combined_gt, combined_hat, k_frac=0.10)
ax.set_title(f'Poisson GLM — Top-10% recovery   Precision = {p10:.3f}')
ax.legend(fontsize=8, markerscale=2); ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout(); plt.show()


## 8. Ridge vs Poisson — side-by-side comparison

Fill in `viability_weights_hat` / `selectivity_weights_hat` from `first_ML_model.ipynb` (Ridge CV) and compare.

In [ ]:
# ── Expected: viability_weights_hat, selectivity_weights_hat from Ridge CV ────
# (run first_ML_model.ipynb first)

methods = {
    'Ridge (CV)' : (viability_weights_hat,   selectivity_weights_hat),
    'Poisson GLM': (viability_weights_poisson, selectivity_weights_poisson),
}

print(f'  Method          r_viab(scores)   r_sel(scores)   Prec@10%')
print(f'  ─────────────────────────────────────────────────────────')
for name, (wv, ws) in methods.items():
    vh   = calculate_scores(sequences, wv)
    sh   = calculate_scores(sequences, ws)
    comb = np.array(vh) / T_viab + np.array(sh) / T_sel
    rv   = pearson(v_gt, vh)
    rs   = pearson(s_gt, sh)
    p10  = precision_at_k(combined_gt, comb)
    print(f'  {name:<15}   {rv:>+.4f}          {rs:>+.4f}         {p10:.3f}')
